# CareScribe: train on a hosted GPU

Use this notebook if your laptop has no GPU PyTorch can use. It runs **your own scripts from your own repository** on a free Colab T4, so everything you built and tested locally stays the source of truth. Nothing here replaces your code.

Before you start: **Runtime → Change runtime type → T4 GPU**. A free runtime can end without warning, so checkpoint often and run the resume drill here too.

In your spec (section 6) and your compute log, the machine is this one: a hosted Tesla T4, 15 GB, compute capability 7.5, so the precision flag is `fp16=True`, not `bf16`.

In [ ]:
!nvidia-smi

## 1. Get your repository

Paste your repository URL. For a private repository, create a fine-grained GitHub token with read and write access to this one repository and paste it when asked. It is not saved in the notebook.

In [ ]:
import getpass, os
REPO = "https://github.com/<your-username>/carescribe.git"   # <- change this
token = getpass.getpass("GitHub token (leave empty for a public repo): ")
url = REPO.replace("https://", f"https://{token}@") if token else REPO
!git clone -q {url} carescribe
%cd carescribe
!git config user.email "you@example.com" && git config user.name "Your Name"   # <- change these

## 2. Install the pinned libraries

This reinstalls PyTorch at the course's pinned version, which takes a few minutes. Restart the runtime if Colab asks you to, then run the cells from section 2 again (not section 1).

In [ ]:
!pip install -q -r requirements-cuda.txt -c constraints.txt
!python src/check_env.py

## 3. Rebuild your dataset here

`data/dataset/` is not in Git, so rebuild it with the same scripts you ran locally. Same code and same seed give you the same files, and you can prove it: the DVC hash printed below must match the one in your committed `data/dataset.dvc`. Change the script names to yours.

In [ ]:
!python src/scrub.py
!python src/build_dataset.py
!python src/phi_scan.py data/dataset/*.jsonl
!dvc add data/dataset -q && cat data/dataset.dvc && git status --short data/

## 4. Smoke test, then the real run

Use a config written for this machine: QLoRA (4-bit), `fp16`, and the base model from your spec. Log the hardware, the wall-clock hours and the peak GPU memory of every run, the smoke test included.

In [ ]:
!python src/train.py configs/<your-config>.yaml --smoke

In [ ]:
!python src/train.py configs/<your-config>.yaml

If the runtime disconnects, run sections 1 to 3 again and then resume:

`!python src/train.py configs/<your-config>.yaml --resume`

Checkpoints live on the runtime's disk and vanish with it. To survive a disconnect, mount Google Drive and point your `output_dir` at a folder there.

## 5. Evaluate, then bring the results home

Run your baseline and held-out evaluation here as well, since they load the model. Then commit what the project asks for: the adapter, the training logs, the evaluation outputs and the exported run history. Do not commit checkpoints.

In [ ]:
!python src/evaluate_model.py configs/<your-config>.yaml --split validation
# ... your remaining evaluation commands ...
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.search_runs(search_all_experiments=True).to_csv("mlflow_runs.csv", index=False)

In [ ]:
!git add runs/*/adapter logs eval mlflow_runs.csv data/dataset.dvc
!git commit -q -m "Training and evaluation runs from Colab T4" && git push